In [0]:
#Look at the data
CATALOG = "capstone_supply_chain"

df = spark.table(f"{CATALOG}.bronze.raw_quality_inspections")
df.printSchema()
df.show(truncate=False, vertical=True)

In [0]:
#Build the Silver table
from pyspark.sql import functions as F

CATALOG = "capstone_supply_chain"

df_qi = spark.table(f"{CATALOG}.bronze.raw_quality_inspections")

# Text where a dollar amount may hide (concat_ws skips NULLs)
money_text = F.concat_ws(" ", F.col("Customer_Impact"), F.col("Detailed_Notes"))

df_silver_qi = (df_qi
    # 1. Types
    .withColumn("inspection_date",
        F.expr("CAST(try_to_timestamp(Inspection_Date, 'yyyy-MM-dd') AS DATE)"))
    .withColumn("sample_size",   F.expr("try_cast(Sample_Size AS INT)"))
    .withColumn("defects_found", F.expr("try_cast(Defects_Found AS INT)"))
    # 2. Calculated fields
    .withColumn("defect_rate_pct",
        F.when(F.col("sample_size") > 0,
               F.round(F.col("defects_found") / F.col("sample_size") * 100, 1)))
    .withColumn("severity_clean", F.initcap(F.trim(F.col("Severity"))))
    .withColumn("severity_rank",
        F.when(F.col("severity_clean") == "Critical", 3)
         .when(F.col("severity_clean") == "Major", 2)
         .when(F.col("severity_clean") == "Minor", 1)
         .otherwise(0))
    .withColumn("status_clean", F.initcap(F.trim(F.col("Status"))))
    .withColumn("is_open", F.col("status_clean") == "Open")
    # 3. Customer impact: NULL or "None ..." means no impact
    .withColumn("has_customer_impact",
        F.when(F.col("Customer_Impact").isNull(), False)
         .when(F.lower(F.trim(F.col("Customer_Impact"))).startswith("none"), False)
         .otherwise(True))
    # 4. First dollar amount mentioned anywhere in the text (NOT always a total, see note)
    .withColumn("cost_mentioned_usd",
        F.expr("try_cast(replace(regexp_extract(concat_ws(' ', Customer_Impact, Detailed_Notes), '\\\\$([0-9,]+)', 1), ',', '') AS DOUBLE)"))
    # 5. Keep clean columns
    .select(
        F.col("Inspection_ID").alias("inspection_id"),
        F.col("PO_ID").alias("po_id"),
        F.col("Part_Number").alias("part_number"),
        "inspection_date",
        F.trim(F.col("Inspector")).alias("inspector"),
        "sample_size", "defects_found", "defect_rate_pct",
        F.trim(F.col("Defect_Type")).alias("defect_type"),
        "severity_clean", "severity_rank",
        F.col("Root_Cause_Analysis").alias("root_cause"),
        F.col("Corrective_Action").alias("corrective_action"),
        "status_clean", "is_open",
        F.col("Customer_Impact").alias("customer_impact"),
        "has_customer_impact", "cost_mentioned_usd",
        F.col("Detailed_Notes").alias("detailed_notes"),
        F.col("_ingested_at").alias("bronze_ingested_at"),
        F.current_timestamp().alias("silver_processed_at"),
    )
    # 6. Data quality
    .filter(F.col("inspection_id").isNotNull())
    .dropDuplicates(["inspection_id"])
)

df_silver_qi.write.mode("overwrite").saveAsTable(f"{CATALOG}.silver.quality_inspections")
print("✅ silver.quality_inspections rows:", df_silver_qi.count())
display(df_silver_qi.select("inspection_id", "po_id", "inspection_date", "defect_rate_pct",
                            "severity_clean", "severity_rank", "status_clean", "is_open",
                            "has_customer_impact", "cost_mentioned_usd"))

In [0]:
#Check your work
qi = spark.table(f"{CATALOG}.silver.quality_inspections")

# 1. Nulls where we need real values (want zeros)
qi.select(
    F.sum(F.col("inspection_date").isNull().cast("int")).alias("null_date"),
    F.sum(F.col("defect_rate_pct").isNull().cast("int")).alias("null_defect_rate"),
).show()

# 2. Open versus closed (the KPI will count the open ones)
qi.groupBy("status_clean", "is_open").count().show()

# 3. Consistency: a defect type should exist exactly when defects were found (want 0 and 0)
print("defects but no type:", qi.where((F.col("defects_found") > 0) & F.col("defect_type").isNull()).count())
print("type but no defects:", qi.where((F.col("defects_found") == 0) & F.col("defect_type").isNotNull()).count())
print("more defects than samples:", qi.where(F.col("defects_found") > F.col("sample_size")).count())

# 4. Does each inspection point to a real PO, for the same part?
po = (spark.table(f"{CATALOG}.silver.purchase_orders")
      .select("po_id", F.col("part_number").alias("po_part_number"), F.col("vendor_name").alias("po_vendor")))
(qi.join(po, "po_id", "left")
   .select("inspection_id", "po_id", "part_number", "po_part_number", "po_vendor",
           (F.col("part_number") == F.col("po_part_number")).alias("part_matches"))
   .show(truncate=False))

In [0]:
#the defect rate, the customer impact flag, and especially cost_mentioned_usd, which uses the tricky regex. Please run this and send me the output:
spark.table("capstone_supply_chain.silver.quality_inspections") \
     .select("inspection_id", "defect_rate_pct", "severity_clean", "severity_rank",
             "has_customer_impact", "cost_mentioned_usd") \
     .orderBy("inspection_id").show(truncate=False)